In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import yaml
from ml_collections.config_dict import ConfigDict

import torch
from sequence_generation.utils import load_seed, load_dataloader, load_generator, load_regressor, expand_simplex, sample_cond_prob_path
from sequence_generation.solver import SimplexEulerSolver

In [ ]:
DNA_ALPHABET = {'A': 0, 'C': 1, 'G': 2, 'T': 3}

In [ ]:
if torch.cuda.is_available():
    device = 'cuda:0'
    print('Using gpu')
else:
    device = 'cpu'
    print('Using cpu.')

In [ ]:
load_seed(42)

# Load dataset/model

In [ ]:
config_file_name = "/nfs/team361/dj16/projects/sequence_generation/configs/enhancer_gosai.yaml"
config_file = yaml.load(open(config_file_name, "r"), yaml.FullLoader)
config = ConfigDict(config_file)

In [ ]:
dirichlet_fm_model, optimizer, lr_scheduler = load_generator(config)

In [ ]:
train_loader, val_loader, test_loader = load_dataloader(config)

# Generate sequence 

In [ ]:
# Load data
for i, batch in enumerate(train_loader):
    print(batch)
    break

In [ ]:
# Expand discrete variable into consinous simplex
xt, alphas = sample_cond_prob_path(config.model, batch['seqs'], config.model.alphabet_size)
prior_pseudocount = 0.1
xt_inp, prior_weights = expand_simplex(xt, alphas, prior_pseudocount)

In [ ]:
xt_inp.shape

In [ ]:
# Run model: get flow
logits = dirichlet_fm_model(seq=xt_inp, t=alphas)

In [ ]:
# Define solver
N = 6
T = torch.linspace(0, 1, N)  # sample times
T = T.to(device=device)

solver = SimplexEulerSolver(config, dirichlet_fm_model)
B, L, _ = xt_inp.shape 
_, _, seq_pred = solver.sample(B,L)

In [ ]:
print(B, L)

In [ ]:
idx_to_base = {v: k for k, v in DNA_ALPHABET.items()}

In [ ]:
sequences = [
    ''.join(idx_to_base[int(i)] for i in row)
    for row in seq_pred
]

for seq in sequences:
    print(seq)
    print()